In [7]:
import os
import math
import re
from urllib.parse import urlparse

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print('All required libraries imported successfully.')


All required libraries imported successfully.


In [8]:
# Suspicious keywords & URL shortening services
SUSPICIOUS_KEYWORDS = [
    'login', 'verify', 'secure', 'account', 'update', 'banking', 'paypal',
    'signin', 'password', 'confirm', 'wallet', 'recover', 'authenticate',
    'billing', 'appleid', 'chase', 'support', 'token', 'access', 'security',
    'alert', 'revalidate', 'ebay', 'amazon-security', 'admin', 'service',
    'bank', 'checkpoint', 'appeal', 'suspend', 'resolution', 'unlock'
]

SHORTENING_SERVICES = {
    'bit.ly', 'tinyurl.com', 't.co', 'goo.gl', 'ow.ly', 'is.gd',
    'buff.ly', 'adf.ly', 'bitly.com', 'bl.ink', 'shorturl.at'
}

def extract_url_features(url: str) -> dict:
    """Extract 26 structural, lexical, statistical and heuristic features."""
    url_clean = str(url).strip()

    if not url_clean.startswith(('http://', 'https://')):
        url_with_scheme = 'http://' + url_clean
    else:
        url_with_scheme = url_clean

    parsed = urlparse(url_with_scheme)
    hostname = parsed.hostname or ''
    path = parsed.path or ''
    query = parsed.query or ''

    # Shannon entropy
    if url_clean:
        probabilities = [url_clean.count(c) / len(url_clean) for c in set(url_clean)]
        entropy = -sum(p * math.log2(p) for p in probabilities if p > 0)
    else:
        entropy = 0.0

    url_lower = url_clean.lower()
    host_lower = hostname.lower()

    suspicious_count = sum(
        1 for keyword in SUSPICIOUS_KEYWORDS if keyword in url_lower
    )

    has_sensitive_keyword = int(any(
        keyword in url_lower
        for keyword in ['login', 'verify', 'secure', 'account', 'banking', 'paypal']
    ))

    is_shortened = int(any(
        service == host_lower or host_lower.endswith('.' + service)
        for service in SHORTENING_SERVICES
    ))

    ip_pattern = r'^(?:\d{1,3}\.){3}\d{1,3}$'
    has_ip = int(bool(re.match(ip_pattern, host_lower)))

    parts = host_lower.split('.') if host_lower else []
    count_subdomains = max(0, len(parts) - 2) if not has_ip else 0

    common_tlds = ['.com', '.net', '.org', '.gov', '.edu']
    tld_in_subdomain = int(
        len(parts) > 2 and any(tld in '.'.join(parts[:-1]) for tld in common_tlds)
    )

    digits_count = sum(c.isdigit() for c in url_clean)
    digits_ratio = digits_count / len(url_clean) if url_clean else 0.0
    special_chars_count = len(re.findall(r'[^a-zA-Z0-9]', url_clean))

    # UCI-style URL length heuristic
    uci_url_length = 1 if len(url_clean) < 54 else (0 if len(url_clean) <= 75 else -1)
    uci_ssl_state = 1 if parsed.scheme == 'https' else -1

    return {
        'url_length': len(url_clean),
        'hostname_length': len(hostname),
        'path_length': len(path),
        'query_length': len(query),
        'count_dots': url_clean.count('.'),
        'count_hyphens': url_clean.count('-'),
        'count_at': url_clean.count('@'),
        'count_question': url_clean.count('?'),
        'count_equal': url_clean.count('='),
        'count_slash': url_clean.count('/'),
        'count_double_slash': int('//' in path),
        'count_subdomains': count_subdomains,
        'count_digits': digits_count,
        'count_special': special_chars_count,
        'digits_ratio': round(digits_ratio, 4),
        'entropy': round(entropy, 4),
        'has_ip': has_ip,
        'has_https': int(parsed.scheme == 'https'),
        'is_shortened': is_shortened,
        'prefix_suffix_hyphen': int('-' in hostname),
        'has_http_in_hostname': int('http' in host_lower),
        'suspicious_words_count': suspicious_count,
        'has_sensitive_keyword': has_sensitive_keyword,
        'tld_in_subdomain': tld_in_subdomain,
        'uci_url_length': uci_url_length,
        'uci_ssl_state': uci_ssl_state
    }

print('Feature extractor defined with 26 feature indicators.')


Feature extractor defined with 26 feature indicators.


In [9]:
# Multi-environment path resolution for loading the dataset
possible_data_paths = [
    os.path.join('data_sets', 'phishing_urls_dataset.csv'),
    os.path.join('training_notebooks', 'data_sets', 'phishing_urls_dataset.csv'),
    os.path.join('..', 'training_notebooks', 'data_sets', 'phishing_urls_dataset.csv'),
    os.path.join(
        os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd(),
        'data_sets',
        'phishing_urls_dataset.csv'
    )
]

data_path = next((p for p in possible_data_paths if os.path.exists(p)), None)

if data_path is None:
    raise FileNotFoundError(
        'Could not find phishing_urls_dataset.csv in candidate paths.'
    )

print(f'Loading dataset from: {os.path.abspath(data_path)}')

df = pd.read_csv(data_path)

print(f'Dataset shape: {df.shape}')
print('\nClass distribution:')
print(df['label'].value_counts().rename({
    0: 'Legitimate (0)',
    1: 'Phishing (1)'
}))

display(df.head())


Loading dataset from: d:\multi-fraud platform\training_notebooks\data_sets\phishing_urls_dataset.csv
Dataset shape: (3000, 28)

Class distribution:
label
Legitimate (0)    1500
Phishing (1)      1500
Name: count, dtype: int64


,url_length,hostname_length,path_length,query_length,count_dots,count_hyphens,count_at,count_question,count_equal,count_slash,...,is_shortened,prefix_suffix_hyphen,has_http_in_hostname,suspicious_words_count,has_sensitive_keyword,tld_in_subdomain,uci_url_length,uci_ssl_state,url,label
0,26,13,5,0,1,0,0,0,0,3,...,0,0,0,0,0,0,1,1,https://bitbucket.org/help,0
1,40,13,10,8,1,0,0,1,1,4,...,1,0,0,0,0,0,1,1,https://microsoft.com/item/view?id=49201,0
2,52,23,13,8,1,1,0,1,1,3,...,0,0,0,0,0,0,1,-1,http://kfh00a0uy9f8p47d9gb.biz/resolve-case?us...,1
3,44,12,24,0,1,0,0,0,0,5,...,0,0,0,0,0,0,1,1,https://theverge.com/questions/tagged/python,0
4,35,6,22,0,1,2,0,0,0,3,...,1,0,0,1,1,0,1,-1,http://bl.ink/wellsfargo-secure-230,1


In [10]:
# Prepare features and labels
feature_cols = [c for c in df.columns if c not in ['url', 'label']]

X = df[feature_cols]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'Training features shape: {X_train.shape}')
print(f'Testing features shape:  {X_test.shape}')
print(f'Number of features:       {len(feature_cols)}')


Training features shape: (2400, 26)
Testing features shape:  (600, 26)
Number of features:       26


In [11]:
# Train Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=3,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print('Random Forest Classifier model training completed successfully.')


Random Forest Classifier model training completed successfully.


In [12]:
# Evaluate on test set
y_pred = rf_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

report = classification_report(
    y_test,
    y_pred,
    target_names=['Legitimate', 'Phishing']
)

# 5-Fold Cross-Validation
cv_scores = cross_val_score(
    rf_model,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print('=' * 55)
print(f'TEST ACCURACY:       {acc * 100:.2f}%')
print(f'5-FOLD CV MEAN ACC:  {cv_scores.mean() * 100:.2f}%')
print(f'5-FOLD CV STD:       {cv_scores.std() * 100:.2f}%')
print('=' * 55)

print(f'\nConfusion Matrix:\n{cm}\n')
print(f'Classification Report:\n{report}')


TEST ACCURACY:       100.00%
5-FOLD CV MEAN ACC:  99.97%
5-FOLD CV STD:       0.07%

Confusion Matrix:
[[300   0]
 [  0 300]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00       300
    Phishing       1.00      1.00      1.00       300

    accuracy                           1.00       600
   macro avg       1.00      1.00      1.00       600
weighted avg       1.00      1.00      1.00       600



In [13]:
# Top 10 Most Predictive Features
importances = rf_model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

print('Top 10 Most Predictive Features for Phishing Detection:')

for rank in range(min(10, len(feature_cols))):
    feat = feature_cols[sorted_idx[rank]]
    score = importances[sorted_idx[rank]]
    print(f'  {rank + 1:2d}. {feat:<25}: {score:.4f}')


Top 10 Most Predictive Features for Phishing Detection:
   1. has_https                : 0.1745
   2. suspicious_words_count   : 0.1195
   3. uci_ssl_state            : 0.1185
   4. digits_ratio             : 0.1055
   5. count_digits             : 0.0916
   6. entropy                  : 0.0816
   7. hostname_length          : 0.0353
   8. count_special            : 0.0327
   9. count_equal              : 0.0284
  10. url_length               : 0.0278


In [14]:
# Export trained model artifact
possible_model_dirs = [
    os.path.join('..', 'backend', 'app', 'modules', 'phishing_website', 'models'),
    os.path.join('backend', 'app', 'modules', 'phishing_website', 'models'),
    os.path.join(
        os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd(),
        '..', 'backend', 'app', 'modules', 'phishing_website', 'models'
    )
]

model_dir = next(
    (p for p in possible_model_dirs if os.path.exists(os.path.dirname(p))),
    possible_model_dirs[0]
)

os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'phishing_model.pkl')

model_artifact = {
    'model': rf_model,
    'feature_cols': feature_cols,
    'accuracy': float(acc),
    'cv_mean_accuracy': float(cv_scores.mean()),
    'classes': ['Legitimate', 'Phishing']
}

joblib.dump(model_artifact, model_path)

print(f'\nModel successfully exported to: {os.path.abspath(model_path)}')



Model successfully exported to: d:\multi-fraud platform\backend\app\modules\phishing_website\models\phishing_model.pkl


In [15]:
# Test Live Inference on Sample URLs
sample_urls = [
    'https://www.google.com',
    'https://github.com/facebook/react',
    'http://secure-login-chase-update-account.fake-domain.com/login',
    'http://paypal-verification-center-security.com/resolve',
    'http://192.168.1.100/webscr/login',
    'https://en.wikipedia.org/wiki/Machine_learning'
]

print('=== LIVE MODEL PREDICTION OUTPUTS ===\n')

for url in sample_urls:
    feats = extract_url_features(url)
    features_df = pd.DataFrame([feats])[feature_cols]

    probabilities = rf_model.predict_proba(features_df)[0]
    predicted_class = int(rf_model.predict(features_df)[0])

    predicted_label = 'Phishing' if predicted_class == 1 else 'Legitimate'
    phishing_probability = probabilities[1]
    confidence = probabilities[predicted_class]

    print(f'URL: {url}')
    print(f' -> Predicted Label:    {predicted_label}')
    print(f' -> Confidence Score:   {confidence * 100:.2f}%')
    print(f' -> Phishing Risk Prob: {phishing_probability * 100:.2f}%')
    print()


=== LIVE MODEL PREDICTION OUTPUTS ===

URL: https://www.google.com
 -> Predicted Label:    Legitimate
 -> Confidence Score:   100.00%
 -> Phishing Risk Prob: 0.00%

URL: https://github.com/facebook/react
 -> Predicted Label:    Legitimate
 -> Confidence Score:   100.00%
 -> Phishing Risk Prob: 0.00%

URL: http://secure-login-chase-update-account.fake-domain.com/login
 -> Predicted Label:    Phishing
 -> Confidence Score:   98.00%
 -> Phishing Risk Prob: 98.00%

URL: http://paypal-verification-center-security.com/resolve
 -> Predicted Label:    Phishing
 -> Confidence Score:   94.00%
 -> Phishing Risk Prob: 94.00%

URL: http://192.168.1.100/webscr/login
 -> Predicted Label:    Phishing
 -> Confidence Score:   76.83%
 -> Phishing Risk Prob: 76.83%

URL: https://en.wikipedia.org/wiki/Machine_learning
 -> Predicted Label:    Legitimate
 -> Confidence Score:   100.00%
 -> Phishing Risk Prob: 0.00%

